In [ ]:
# @title 0) 저장소 클론·pip 설치 (로컬에서는 생략 가능)
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start: Path | None = None) -> Path | None:
    candidate = (start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        print(f"already cloned -> git pull: {workdir}")
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        print(f"cloning {REPO_URL} -> {workdir}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir
else:
    print(f"local repo: {root}")

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("pip install -e . done")


# 03. Layer Sweep Feature Search

질문: bat-and-ball 함정 답을 지지하는 feature 후보가 어느 layer에서 강하게 나타나는가?

학습 포인트:
- layer마다 active SAE feature를 뽑습니다.
- 각 feature 제거 후 `logprob(10 cents) - logprob(5 cents)` margin이 얼마나 줄어드는지 비교합니다.
- 가장 강한 feature handle을 `outputs/candidates/bat_ball_top_feature.json`에 저장해 후속 노트북에서 재사용합니다.

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_MODEL_ID,
    DEFAULT_QWEN_SCOPE_REPO_ID,
    LureCase,
    answer_logprob_margin,
    answer_variant_rows,
    bat_ball_answer_variants,
    bat_ball_paraphrases,
    candidate_feature_rows,
    case_transfer_rows,
    coefficient_sweep_for_handle,
    control_delta_bypass_rows,
    crt_transfer_cases,
    decoder_cosine_rows,
    default_sae_device,
    dtype_from_name,
    feature_handle_from_result,
    intervention_mode_rows,
    layer_feature_search_rows,
    load_or_discover_handle_and_sae,
    load_qwen_language_model,
    load_qwen_scope_sae,
    prompt_token_window_rows,
    rank_lure_feature_effects,
    recommended_dtype_name,
    sae_decoder_direction,
    save_feature_handle,
    semantic_lure_cases,
    token_position_sweep_rows,
)

MODEL_ID = DEFAULT_MODEL_ID
SAE_REPO_ID = DEFAULT_QWEN_SCOPE_REPO_ID
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE
HANDLE_CACHE = root / "outputs" / "candidates" / "bat_ball_top_feature.json"

lm = load_qwen_language_model(MODEL_ID, device_map="auto", dtype=DTYPE, dispatch=True)
print({"model": MODEL_ID, "sae_repo": SAE_REPO_ID, "dtype": DTYPE, "sae_device": SAE_DEVICE})


In [ ]:
CASE = BAT_BALL_CASE
LAYERS = [6, 14, 21, 27]
TOP_N = 8

sae_by_layer = {
    layer: load_qwen_scope_sae(
        SAE_REPO_ID,
        layer,
        device=SAE_DEVICE,
        dtype=dtype_from_name(SAE_DTYPE),
    )
    for layer in LAYERS
}

rows = layer_feature_search_rows(
    lm,
    CASE,
    layers=LAYERS,
    sae_by_layer=sae_by_layer,
    top_n=TOP_N,
    coefficient=1.0,
    intervention_mode="remove_activation",
)
display(rows[:40])

best_row = rows[0]


class _Result:
    pass


best = _Result()
for key, value in best_row.items():
    setattr(best, key, value)
best.layer = best_row["layer"]
best.feature_id = best_row["feature_id"]
best.feature_value = best_row["feature_value"]
best.margin_delta = best_row["margin_delta"]
best.baseline_margin = best_row["baseline_margin"]
best.ablated_margin = best_row["ablated_margin"]
best.intervention_mode = best_row["intervention_mode"]
best.coefficient = best_row["coefficient"]
handle = feature_handle_from_result(CASE, best)
save_feature_handle(handle, HANDLE_CACHE)
print("saved feature handle:", HANDLE_CACHE)
display(handle.as_row())


해석 체크: `margin_delta`가 큰 layer-feature를 다음 실험의 기본 후보로 삼습니다. 같은 feature가 여러 prompt 변형에서도 효과를 보이는지는 07/10/13번에서 확인합니다.